# Инициализация

Загружаем библиотеки необходимые для выполнения кода ноутбука.

In [1]:
import pandas as pd
import numpy as np
from IPython.display import display

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)


# === ЭТАП 1 ===

# Загрузка первичных данных

Загружаем первичные данные из файлов:
- tracks.parquet
- catalog_names.parquet
- interactions.parquet

In [2]:
tracks = pd.read_parquet("tracks.parquet")
catalog = pd.read_parquet("catalog_names.parquet")
interactions = pd.read_parquet("interactions.parquet")

print(tracks.shape, catalog.shape, interactions.shape)



(1000000, 4) (1812471, 3) (222629898, 4)


In [3]:
display(tracks.head(5))
display(catalog.head(5))
display(interactions.head(5))

,track_id,albums,artists,genres
0,26,"[3, 2490753]",[16],"[11, 21]"
1,38,"[3, 2490753]",[16],"[11, 21]"
2,135,"[12, 214, 2490809]",[84],[11]
3,136,"[12, 214, 2490809]",[84],[11]
4,138,"[12, 214, 322, 72275, 72292, 91199, 213505, 24...",[84],[11]


,id,type,name
0,3,album,Taller Children
1,12,album,Wild Young Hearts
2,13,album,Lonesome Crow
3,17,album,Graffiti Soul
4,26,album,Blues Six Pack


,user_id,track_id,track_seq,started_at
0,0,99262,1,2022-07-17
1,0,589498,2,2022-07-19
2,0,590262,3,2022-07-21
3,0,590303,4,2022-07-22
4,0,590692,5,2022-07-22


Уменьшаем кол-во слушателей для облегчения нагрузки на ядро

In [8]:
np.random.seed(42)

users_sample = (
    interactions[["user_id"]]
    .drop_duplicates()
    .sample(n=200_000, random_state=42)
)

interactions_small = interactions.merge(users_sample, on="user_id", how="inner")

interactions_small = (
    interactions_small
    .sort_values(["user_id", "started_at"])
    .groupby("user_id", as_index=False)
    .head(200)
)


interactions_small.shape
interactions_small["user_id"].nunique()



200000

In [10]:
n_users = interactions_small["user_id"].nunique()
n_tracks_events = interactions_small["track_id"].nunique()
n_tracks_items = tracks["track_id"].nunique()

print("unique users in interactions_small:", n_users)
print("unique tracks in interactions_small:", n_tracks_events)
print("unique tracks in tracks.parquet:", n_tracks_items)


unique users in interactions_small: 200000
unique tracks in interactions_small: 644822
unique tracks in tracks.parquet: 1000000


In [11]:
interactions = interactions_small.copy()

# Обзор данных

Проверяем данные, есть ли с ними явные проблемы.

In [12]:
tracks.info()
print('============')
catalog.info()
print('============')
interactions.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000000 entries, 0 to 999999
Data columns (total 4 columns):
 #   Column    Non-Null Count    Dtype 
---  ------    --------------    ----- 
 0   track_id  1000000 non-null  int64 
 1   albums    1000000 non-null  object
 2   artists   1000000 non-null  object
 3   genres    1000000 non-null  object
dtypes: int64(1), object(3)
memory usage: 30.5+ MB
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1812471 entries, 0 to 1812470
Data columns (total 3 columns):
 #   Column  Dtype 
---  ------  ----- 
 0   id      int64 
 1   type    object
 2   name    object
dtypes: int64(1), object(2)
memory usage: 41.5+ MB
<class 'pandas.core.frame.DataFrame'>
Index: 17023467 entries, 0 to 32278239
Data columns (total 4 columns):
 #   Column      Dtype         
---  ------      -----         
 0   user_id     int32         
 1   track_id    int32         
 2   track_seq   int16         
 3   started_at  datetime64[ns]
dtypes: datetime64[ns](1), int16(1

In [24]:
interactions["track_id"] = interactions["track_id"].astype("int64")
interactions["user_id"] = interactions["user_id"].astype("int64")


In [13]:
def quick_checks(df: pd.DataFrame, name: str, dup_subset=None, null_top: int = 20) -> None:
    """
    Быстрая проверка датасета: размер, дубликаты, пропуски.
    
    dup_subset:
        - None -> пытаемся по всем колонкам, но если не получится (списки/массивы) — автоматически берём только hashable-колонки
        - list[str] -> считаем дубликаты по указанным колонкам
    """
    print(f"=== {name} ===")
    print("shape:", df.shape)

    # Дубликаты
    if dup_subset is not None:
        dup_cnt = df.duplicated(subset=dup_subset).sum()
        print(f"duplicates (subset={dup_subset}):", int(dup_cnt))
    else:
        try:
            dup_cnt = df.duplicated().sum()
            print("duplicates (all columns):", int(dup_cnt))
        except TypeError:
            # Авто-режим: берём только колонки без list/ndarray/set/dict
            def is_hashable_col(s: pd.Series) -> bool:
                x = s.dropna()
                if x.empty:
                    return True
                v = x.iloc[0]
                return not isinstance(v, (list, tuple, set, dict, np.ndarray))

            safe_cols = [c for c in df.columns if is_hashable_col(df[c])]
            dup_cnt = df.duplicated(subset=safe_cols).sum()
            print(f"duplicates (safe columns={safe_cols}):", int(dup_cnt))

    # Пропуски
    nulls = df.isna().sum().sort_values(ascending=False)
    print("nulls (top):")
    display(nulls.head(null_top))
    print()


quick_checks(tracks, "tracks")
quick_checks(catalog, "catalog_names")
quick_checks(interactions, "interactions")


=== tracks ===
shape: (1000000, 4)
duplicates (safe columns=['track_id']): 0
nulls (top):


track_id    0
albums      0
artists     0
genres      0
dtype: int64


=== catalog_names ===
shape: (1812471, 3)
duplicates (all columns): 0
nulls (top):


id      0
type    0
name    0
dtype: int64


=== interactions ===
shape: (17023467, 4)
duplicates (all columns): 0
nulls (top):


user_id       0
track_id      0
track_seq     0
started_at    0
dtype: int64

In [14]:
tracks[["albums","artists","genres"]].head()

def explode_unique_ids(df, col):
    s = df[col].explode()
    s = s.dropna()
    return set(s.unique())

album_ids = explode_unique_ids(tracks, "albums")
artist_ids = explode_unique_ids(tracks, "artists")
genre_ids = explode_unique_ids(tracks, "genres")

print(len(album_ids), len(artist_ids), len(genre_ids))



658724 153581 173


In [15]:
catalog_albums = set(catalog.loc[catalog["type"] == "album", "id"].unique())
catalog_artists = set(catalog.loc[catalog["type"] == "artist", "id"].unique())
catalog_genres = set(catalog.loc[catalog["type"] == "genre", "id"].unique())

missing_albums = album_ids - catalog_albums
missing_artists = artist_ids - catalog_artists
missing_genres = genre_ids - catalog_genres

print("Missing albums:", len(missing_albums))
print("Missing artists:", len(missing_artists))
print("Missing genres:", len(missing_genres))


Missing albums: 0
Missing artists: 0
Missing genres: 30


In [17]:
print("Example missing_genres:", list(missing_genres)[:10])


Example missing_genres: [130, 131, 132, 133, 134, 135, 146, 148, 150, 151]


In [23]:
tracks_with_missing_genres = tracks[
    tracks["genres"].apply(lambda g: any(x in missing_genres for x in g))
]

tracks_with_missing_genres.shape
tracks_with_missing_genres.head()
tracks_with_missing_genres["track_id"].nunique()


48345

In [25]:
tracks_with_missing_genres["track_id"].nunique()


48345

In [26]:
catalog = pd.concat(
    [
        catalog,
        pd.DataFrame({
            "id": list(missing_genres),
            "type": "genre",
            "name": "unknown_genre"
        })
    ],
    ignore_index=True
)


# Выводы

Приведём выводы по первому знакомству с данными:
- есть ли с данными явные проблемы,
- какие корректирующие действия (в целом) были предприняты.

Были загружены данные о треках, справочниках и пользовательских взаимодействиях.
Количество уникальных пользователей (около 1,4 млн) и треков (1 млн) соответствует условиям задачи. Пропусков и дубликатов в данных не обнаружено.

Типы данных в целом заданы корректно. Идентификаторы пользователей и треков приведены к единому типу для согласованности между таблицами. В таблице tracks поля с альбомами, артистами и жанрами представлены списками идентификаторов, что было учтено при проверке данных.

В ходе проверки справочников было выявлено, что в catalog_names отсутствуют описания для части жанров. Эти жанры используются в данных о треках и относятся примерно к 48 тыс. треков. Для сохранения целостности данных недостающие жанры были добавлены в справочник с техническим названием unknown_genre.

После выполненных корректировок данные можно считать подготовленными для дальнейших этапов работы.

# === ЭТАП 2 ===

# EDA

Распределение количества прослушанных треков.

Наиболее популярные треки

Наиболее популярные жанры

Треки, которые никто не прослушал

# Преобразование данных

Преобразуем данные в формат, более пригодный для дальнейшего использования в расчётах рекомендаций.

# Сохранение данных

Сохраним данные в двух файлах в персональном S3-бакете по пути `recsys/data/`:
- `items.parquet` — все данные о музыкальных треках,
- `events.parquet` — все данные о взаимодействиях.

# Очистка памяти

Здесь, может понадобится очистка памяти для высвобождения ресурсов для выполнения кода ниже. 

Приведите соответствующие код, комментарии, например:
- код для удаление более ненужных переменных,
- комментарий, что следует перезапустить kernel, выполнить такие-то начальные секции и продолжить с этапа 3.

# === ЭТАП 3 ===

# Загрузка данных

Если необходимо, то загружаем items.parquet, events.parquet.

# Разбиение данных

Разбиваем данные на тренировочную, тестовую выборки.

# Топ популярных

Рассчитаем рекомендации как топ популярных.

# Персональные

Рассчитаем персональные рекомендации.

# Похожие

Рассчитаем похожие, они позже пригодятся для онлайн-рекомендаций.

# Построение признаков

Построим три признака, можно больше, для ранжирующей модели.

# Ранжирование рекомендаций

Построим ранжирующую модель, чтобы сделать рекомендации более точными. Отранжируем рекомендации.

# Оценка качества

Проверим оценку качества трёх типов рекомендаций: 

- топ популярных,
- персональных, полученных при помощи ALS,
- итоговых
  
по четырем метрикам: recall, precision, coverage, novelty.

# === Выводы, метрики ===

Основные выводы при работе над расчётом рекомендаций, рассчитанные метрики.